## 1. 长序列带来的 O(n²) 噩梦

当序列长度 n 变大时，`Q @ K.T` 这个矩阵乘法的代价是 O(n² d_k) 计算量和 O(n²) 显存。一张 32GB 的 GPU，用 float32 时，n=2048 的 `attn_weights` 就占 16 MB，n=8192 直接爆炸到 256 MB。但一个句子的主谓宾往往仅相隔几个词，远距离的依赖其实是稀疏的。研究者开始思考：**能不能让每个查询只与一部分键交互，从而跳开 n² 诅咒？**


## 2. 稀疏注意力：滑动窗口 —— "单词只与邻居说话"

### 2.1 思路：局部窗口的直觉
人类阅读时，视野只有一个窗口大小，理解一个词时主要依赖周围几个词。假如我们强制让位置 i 只看索引在 [i-w, i+w] 内的键，那么每个查询只需计算 2w+1 个点积，复杂度降为 O(n·w)。

### 2.2 数学表达：用掩码屏蔽远方
原注意力公式：
$$
\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V
$$
我们引入掩码矩阵 $ M \in \{0, -\infty\}^{n \times n} $：
$$
M_{ij} = \begin{cases}
0 & \text{if } |i - j| \le w \\
-\infty & \text{otherwise}
\end{cases}
$$
将其加到缩放点积上，softmax 会自动把 $-\infty$ 位置的权重压为 0：
$$
\text{WindowAttention}(Q,K,V) = \text{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}} + M\right) V
$$


In [1]:
import torch

torch.manual_seed(42)
n, d_model, d_k = 12, 64, 64      # 序列长度12，维度64
window_size = 3                    # 单侧窗口大小

# 随机输入序列
X = torch.randn(n, d_model)

# 构造窗口掩码矩阵 M in {0, 1}^{n x n}
mask = torch.ones(n, n)           # 先假设所有位置都可见
for i in range(n):
    for j in range(n):
        if abs(i - j) > window_size:
            mask[i, j] = 0        # 距离超过窗口就屏蔽

# 把 0 变成 -inf，1 变成 0
mask_inf = (1 - mask) * -1e9      # [12, 12]

print("窗口掩码矩阵 M（1=可见, 0=屏蔽）:")
print(mask.numpy().astype(int))
print()
print("mask_inf 前 4 行（0=可见, -1e9=屏蔽）:")
print(mask_inf[:4].numpy().astype(int))


窗口掩码矩阵 M（1=可见, 0=屏蔽）:
[[1 1 1 1 0 0 0 0 0 0 0 0]
 [1 1 1 1 1 0 0 0 0 0 0 0]
 [1 1 1 1 1 1 0 0 0 0 0 0]
 [1 1 1 1 1 1 1 0 0 0 0 0]
 [0 1 1 1 1 1 1 1 0 0 0 0]
 [0 0 1 1 1 1 1 1 1 0 0 0]
 [0 0 0 1 1 1 1 1 1 1 0 0]
 [0 0 0 0 1 1 1 1 1 1 1 0]
 [0 0 0 0 0 1 1 1 1 1 1 1]
 [0 0 0 0 0 0 1 1 1 1 1 1]
 [0 0 0 0 0 0 0 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 1 1 1 1]]

mask_inf 前 4 行（0=可见, -1e9=屏蔽）:
[[          0           0           0           0 -1000000000 -1000000000
  -1000000000 -1000000000 -1000000000 -1000000000 -1000000000 -1000000000]
 [          0           0           0           0           0 -1000000000
  -1000000000 -1000000000 -1000000000 -1000000000 -1000000000 -1000000000]
 [          0           0           0           0           0           0
  -1000000000 -1000000000 -1000000000 -1000000000 -1000000000 -1000000000]
 [          0           0           0           0           0           0
            0 -1000000000 -1000000000 -1000000000 -1000000000 -1000000000]]


### 2.3 实际计算：全注意力 vs 窗口注意力

现在用上一步构造的掩码执行完整的缩放点积注意力，并同时计算两种模式：
- **全注意力**：不加掩码，每个查询与所有键交互，复杂度 O(n²)。
- **窗口注意力**：加上掩码，每个查询只与窗口内的键交互，复杂度 O(n·w)。

通过对比位置 0 在这两种模式下的注意力分布，可以直观看到：全注意力均匀分散到 12 个位置，而窗口注意力将所有权重集中到窗口内的 4 个位置上，其余位置被 softmax 自动置零。最后打印两种模式的实际乘加次数和 n=8192 时的量级对比，让 O(n²) vs O(n·w) 的复杂度差异一目了然。

In [2]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)
n, d_model, d_k = 12, 64, 64
window_size = 3

X = torch.randn(n, d_model)

# 可学习的投影矩阵
W_Q = torch.randn(d_model, d_k, requires_grad=True)
W_K = torch.randn(d_model, d_k, requires_grad=True)
W_V = torch.randn(d_model, d_k, requires_grad=True)

# 计算 Q, K, V 并归一化，使点积稳定在 [-1,1] 范围
Q = F.normalize(X @ W_Q, dim=-1)   # [12, 64]
K = F.normalize(X @ W_K, dim=-1)   # [12, 64]
V = X @ W_V                        # [12, 64]

# ---------- 构造窗口掩码 ----------
mask = torch.ones(n, n)
for i in range(n):
    for j in range(n):
        if abs(i - j) > window_size:
            mask[i, j] = 0
mask_inf = (1 - mask) * -1e9

# ---------- 缩放点积注意力 ----------
scale = torch.sqrt(torch.tensor(d_k, dtype=torch.float))
scores = (Q @ K.T) / scale          # [12, 12]

# 全注意力（无掩码）
attn_full = F.softmax(scores, dim=-1)

# 窗口注意力（加掩码）
scores_masked = scores + mask_inf
attn_window = F.softmax(scores_masked, dim=-1)
output = attn_window @ V            # [12, 64]

# 对比输出
print("全注意力——所有12个位置都有权重:")
print(attn_full[0].detach().numpy().round(3))
print("窗口注意力——只关注位置0~3:")
print(attn_window[0].detach().numpy().round(3))
print()
print("计算量对比:")
print(f"  全注意力: Q @ K.T 需要 {n}x{n}x{d_k} = {n*n*d_k} 次乘加")
print(f"  窗口注意力: 每个查询只算 {2*window_size+1} 个键，共 {n}x{2*window_size+1}x{d_k} = {n*(2*window_size+1)*d_k} 次乘加")
print(f"  当 n 很大时（如 n=8192），全注意力 O(n^2) = {8192*8192} 项，窗口注意力 O(n.w) = {8192*(2*3+1)} 项")


全注意力——所有12个位置都有权重:
[0.083 0.082 0.085 0.084 0.081 0.084 0.082 0.083 0.085 0.082 0.084 0.084]
窗口注意力——只关注位置0~3:
[0.249 0.246 0.254 0.251 0.    0.    0.    0.    0.    0.    0.    0.   ]

计算量对比:
  全注意力: Q @ K.T 需要 12x12x64 = 9216 次乘加
  窗口注意力: 每个查询只算 7 个键，共 12x7x64 = 5376 次乘加
  当 n 很大时（如 n=8192），全注意力 O(n^2) = 67108864 项，窗口注意力 O(n.w) = 57344 项


**说明**：  
- `mask` 用双重循环按条件填入 0/1，虽然简单但体现了规则。  
- `mask_inf` 用 `(1 - mask) * -1e9` 将不可见位置设成一个极大的负数，加法后变成负无穷。  
- softmax 遇到 `-1e9` 就输出 0，相当于彻底忽略这些键。  
- 虽然我们仍构造了完整的 `scores` 矩阵，但实际优化实现（如块稀疏内核）只计算窗口内项，这里只做逻辑展示。

### 2.4 立即得到的收益
- 参数量未变，但每个查询的计算量从 n 次点积降为 2w+1。  
- 长文本训练显存大幅下降，n 很大时 w 固定，计算复杂度近线性。  
- 代价：丢失了长距离直接交互的能力，某些需要远距离关联的任务会受到影响。


## 3. 混合注意力：局部窗口 + 全局 Token —— "开一扇小天窗"

### 3.1 思路：用少数全局节点搭桥
纯粹的局部窗口让序列两端的词永不见面。解决办法是引入少量**全局 Token**，它们能被所有位置看见，也能看见所有位置，充当信息的中转站。这些 Token 可以是从数据中学到的向量，就像 BERT 的 `[CLS]` 一样。

### 3.2 数学表达：可学习的全局向量与混合掩码
设原始序列长度为 n，在前面添加 g 个可学习全局向量 $ G \in \mathbb{R}^{g \times d_{\text{model}}} $。总序列长度 $ N = n+g $。定义掩码规则：
- 对于全局 Token 行 (0~g-1)：整行全是 1，可以注意任何位置。
- 对于局部 Token 行 (g~N-1)：只能在局部范围 [i-w, i+w] 内的**其他局部 Token** 上保持可见，但全局 Token 列（0~g-1）永远可见。

掩码矩阵 $ M $ 中：
$$
M_{ij} = \begin{cases}
0 & \text{if } i < g \text{ (全局行)} \\
0 & \text{if } j < g \text{ (全局列，任何行可见)} \\
0 & \text{if } i \ge g, j \ge g \text{ 且 } | (i-g) - (j-g) | \le w \\
-\infty & \text{otherwise}
\end{cases}
$$


In [3]:
import torch

torch.manual_seed(42)
n, d_model, d_k = 12, 64, 64
window_size = 3
g = 2                     # 2个全局 token

# 原始局部序列 + 全局 token
X_local = torch.randn(n, d_model)                         # [12,64]
global_tokens = torch.randn(g, d_model, requires_grad=True)  # [2,64]
X = torch.cat([global_tokens, X_local], dim=0)           # [14,64]

total_len = n + g
mask = torch.ones(total_len, total_len)   # 初始全可见

# 对局部 token 之间应用窗口限制
for i in range(g, total_len):
    for j in range(g, total_len):
        if abs((i - g) - (j - g)) > window_size:
            mask[i, j] = 0

print("混合注意力掩码矩阵（1=可见, 0=屏蔽）:")
print(mask.numpy().astype(int))
print()
print("说明:")
print("  - 第0~1行（全局 token）: 全1，可以注意所有位置")
print("  - 第0~1列（全局 token）: 全1，所有行都能注意全局 token")
print("  - 第2~13行（局部 token）: 只在窗口内注意其他局部 token")


混合注意力掩码矩阵（1=可见, 0=屏蔽）:
[[1 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 0 0 0 0 0 0 0 0]
 [1 1 1 1 1 1 1 0 0 0 0 0 0 0]
 [1 1 1 1 1 1 1 1 0 0 0 0 0 0]
 [1 1 1 1 1 1 1 1 1 0 0 0 0 0]
 [1 1 0 1 1 1 1 1 1 1 0 0 0 0]
 [1 1 0 0 1 1 1 1 1 1 1 0 0 0]
 [1 1 0 0 0 1 1 1 1 1 1 1 0 0]
 [1 1 0 0 0 0 1 1 1 1 1 1 1 0]
 [1 1 0 0 0 0 0 1 1 1 1 1 1 1]
 [1 1 0 0 0 0 0 0 1 1 1 1 1 1]
 [1 1 0 0 0 0 0 0 0 1 1 1 1 1]
 [1 1 0 0 0 0 0 0 0 0 1 1 1 1]]

说明:
  - 第0~1行（全局 token）: 全1，可以注意所有位置
  - 第0~1列（全局 token）: 全1，所有行都能注意全局 token
  - 第2~13行（局部 token）: 只在窗口内注意其他局部 token


### 3.3 实际计算：混合注意力中的注意力分布

下面用上一步构造的混合掩码执行注意力计算，并观察三个有代表性的位置：
- **全局 token 0**（索引 0）：应能注意到全部 14 个位置，证明全局行不受限制。
- **局部 token 2**（索引 2，即原始序列第 0 个词）：序列开头的词，其窗口内邻居只有右侧（索引 3~5），左侧没有词。此外它必须能看到两个全局 token（索引 0~1）。
- **局部 token 10**（索引 10，即原始序列第 8 个词）：序列中部的词，窗口内邻居左右各有若干个，同时也必须看到两个全局 token。

通过对比这三者的非零位置集合，可以直观验证混合掩码的规则是否生效：**每个局部 token 的注意力都包含全局列（索引 0~1）加上它自己的局部窗口邻居**，而窗口外的局部位置被正确屏蔽。这正是 "开一扇小天窗" 的核心——全局 token 充当信息中转站，让原本被窗口隔开的远距离局部 token 可以间接通信。

In [5]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)
n, d_model, d_k = 12, 64, 64
window_size = 3
g = 2                     # 2个全局 token

X_local = torch.randn(n, d_model)                         # [12,64]
global_tokens = torch.randn(g, d_model, requires_grad=True)  # [2,64]
X = torch.cat([global_tokens, X_local], dim=0)           # [14,64]

W_Q = torch.randn(d_model, d_k)
W_K = torch.randn(d_model, d_k)
W_V = torch.randn(d_model, d_k)

# 投影并归一化
Q = F.normalize(X @ W_Q, dim=-1)   # [14,64]
K = F.normalize(X @ W_K, dim=-1)   # [14,64]
V = X @ W_V                        # [14,64]

total_len = n + g
mask = torch.ones(total_len, total_len)
for i in range(g, total_len):
    for j in range(g, total_len):
        if abs((i - g) - (j - g)) > window_size:
            mask[i, j] = 0
mask_inf = (1 - mask) * -1e9

scale = torch.sqrt(torch.tensor(d_k, dtype=torch.float))
scores = (Q @ K.T) / scale
scores_masked = scores + mask_inf
attn_weights = F.softmax(scores_masked, dim=-1)
output = attn_weights @ V   # [14,64]

# 观察不同位置的注意力分布
aw = attn_weights.detach().numpy()

print("全局 token 0 的注意力分布（所有位置都有权重）:")
print(aw[0].round(3))
print(f"  非零权重数: {(aw[0] > 1e-5).sum()}")
print()

print("局部 token 2（即原始序列第0个）的注意力分布:")
print(aw[2].round(3))
nonzeros = [str(i) for i, v in enumerate(aw[2]) if v > 1e-5]
print(f"  非零位置: {', '.join(nonzeros)}")
print(f"  -> 全局列(0,1) + 局部窗口内邻居(2~5)")
print()

print("局部 token 10（即原始序列第8个）的注意力分布:")
print(aw[10].round(3))
nonzeros2 = [str(i) for i, v in enumerate(aw[10]) if v > 1e-5]
print(f"  非零位置: {', '.join(nonzeros2)}")
print(f"  -> 全局列(0,1) + 局部窗口内邻居(7~13)")


全局 token 0 的注意力分布（所有位置都有权重）:
[0.071 0.071 0.072 0.073 0.073 0.071 0.07  0.07  0.07  0.071 0.07  0.072
 0.073 0.072]
  非零权重数: 14

局部 token 2（即原始序列第0个）的注意力分布:
[0.163 0.169 0.168 0.17  0.165 0.165 0.    0.    0.    0.    0.    0.
 0.    0.   ]
  非零位置: 0, 1, 2, 3, 4, 5
  -> 全局列(0,1) + 局部窗口内邻居(2~5)

局部 token 10（即原始序列第8个）的注意力分布:
[0.111 0.11  0.    0.    0.    0.    0.    0.111 0.112 0.111 0.112 0.111
 0.111 0.111]
  非零位置: 0, 1, 7, 8, 9, 10, 11, 12, 13
  -> 全局列(0,1) + 局部窗口内邻居(7~13)


**逐行解释**：  
- `global_tokens` 是 `requires_grad=True` 的参数，会随着训练更新。  
- 掩码构造只对局部 token 之间的交互加窗口限制，全局 token 完全不受限。  
- 局部 token 始终可以看到全局 token，这保证任何位置的局部信息都能通过全局 token 扩散到远处。  
- 输出 `attn_weights @ V` 仍然是一个常规的线性组合，后续处理无需改变。

### 3.4 效果与代价
- 复杂度：每个局部查询要处理 g 个全局键 + 2w+1 个局部键，远小于 n。  
- 全局 Token 数量很小（例如 1~8 个），计算开销微乎其微。  
- 这种设计被 Longformer、BigBird 等模型采用，能很好处理摘要、QA 等长文本任务。


## 4. 线性注意力：代数恒等式绕开 n×n 矩阵

### 4.1 思路：如果相似度可以分解为两个向量的内积
标准注意力对相似度做了指数运算： $ \text{sim}(q, k) = \exp(q \cdot k / \sqrt{d}) $，这无法拆解成两个独立向量的点积，因此我们被迫先算出 n×n 的相似矩阵。但如果我们换一个相似度函数，使得 $ \text{sim}(q, k) = \phi(q)^\top \phi(k) $，那么就可以利用矩阵乘法的结合律，把计算顺序从 `(Q K^T) V` 变成 `Q (K^T V)`，从而避免出现 n×n 的中间矩阵。

### 4.2 推导：从结合律到线性复杂度
原注意力：
$$
\text{Attention}(Q,K,V) = \frac{\sum_j \exp(q_i^\top k_j / \sqrt{d}) v_j}{\sum_j \exp(q_i^\top k_j / \sqrt{d})}
$$
用 $\phi$ 替代指数，得到近似：
$$
\text{Attention}_{\text{lin}}(Q,K,V) = \frac{\sum_j \phi(q_i)^\top \phi(k_j) v_j}{\sum_j \phi(q_i)^\top \phi(k_j)}
$$
提取 $\phi(q_i)$：
$$
= \frac{\phi(q_i)^\top \sum_j \phi(k_j) v_j^\top}{\phi(q_i)^\top \sum_j \phi(k_j)}
$$
定义两个聚合项（遍历所有 j 计算，与 i 无关）：
$$
C = \sum_j \phi(k_j) v_j^\top \quad (\text{形状 } d \times d)
$$
$$
s = \sum_j \phi(k_j) \quad (\text{形状 } d)
$$
那么对于每个查询 i：
$$
\text{output}_i = \frac{\phi(q_i)^\top C}{\phi(q_i)^\top s}
$$
整个计算过程：先 O(nd²) 算 C 和 s，再 O(nd²) 算所有输出，总复杂度 O(nd²)。当 n ≫ d（如 d=64, n=4096），O(nd²) 远小于 O(n²d)，近乎线性。

### 4.3 数学表达式（最终）
选定特征映射 $\phi(x) = \text{ELU}(x) + 1$（保证非负，近似指数性质）：
$$
\text{LinearAttention}(Q, K, V) = \frac{ (\phi(Q) \; (\phi(K)^\top V)) }{ \phi(Q) \; (\phi(K)^\top \mathbf{1}) }
$$
其中除号表示逐元素相除，$\mathbf{1}$ 是全 1 列向量。


In [6]:
def linear_attention(Q, K, V, eps=1e-6):
    """
    参数:
        Q, K, V: [n, d] 张量
    返回:
        output: [n, d] 线性注意力结果
    """
    # 1. 特征映射：elu + 1，保证非负
    phi_Q = F.elu(Q) + 1   # [n, d]
    phi_K = F.elu(K) + 1   # [n, d]

    # 2. 构建上下文矩阵 C = phi(K)^T V，形状 [d, d]
    #    phi_K 转置 [d, n] 乘 V [n, d] -> [d, d]
    C = phi_K.T @ V

    # 3. 归一化项 s = phi(K)^T @ 1，形状 [d]
    #    全1向量 [n, 1] 乘 phi_K.T [d, n] -> [d, 1] 然后 squeeze
    s = phi_K.T @ torch.ones(K.shape[0], 1, device=K.device)  # [d, 1]

    # 4. 分子: phi_Q @ C，形状 [n, d]
    num = phi_Q @ C

    # 5. 分母: phi_Q @ s，形状 [n, 1]
    den = phi_Q @ s + eps   # 避免除零

    # 6. 逐元素相除
    output = num / den
    return output

# ---------- 对比标准注意力 ----------
torch.manual_seed(42)
n_test, d_test = 8, 64
Q_test = torch.randn(n_test, d_test)
K_test = torch.randn(n_test, d_test)
V_test = torch.randn(n_test, d_test)

# 标准 softmax 注意力
scale = torch.sqrt(torch.tensor(d_test, dtype=torch.float))
scores_std = Q_test @ K_test.T / scale
attn_std = F.softmax(scores_std, dim=-1)
out_std = attn_std @ V_test

# 线性注意力
out_lin = linear_attention(Q_test, K_test, V_test)

print("标准注意力输出形状:", out_std.shape)
print("线性注意力输出形状:", out_lin.shape)
print("两个输出的均方误差 (MSE):", torch.mean((out_std - out_lin)**2).item())


标准注意力输出形状: torch.Size([8, 64])
线性注意力输出形状: torch.Size([8, 64])
两个输出的均方误差 (MSE): 0.11201479285955429


**代码细解：**  
- `F.elu(Q) + 1`：ELU 在负半轴指数衰减到 -1，加 1 后始终为正，近似 exp 但可分解。  
- `C = phi_K.T @ V`：这是整个算法的核心——把键值对压缩成一个 d×d 矩阵，**序列长度 n 在这里消失了**。  
- `s = phi_K.T @ ones`：归一项，对应于 softmax 的分母。  
- `num / den`：最终每个查询得到一个加权平均，无需计算 n×n 矩阵。

### 4.5 复杂度分析与局限性
**复杂度对比：**
- 标准注意力：`Q @ K.T` 是 O(n²d)，softmax 及乘法 O(n²d)，内存 O(n²)。
- 线性注意力：`phi_K.T @ V` 是 O(nd²)，`phi_Q @ C` 是 O(nd²)，内存 O(nd + d²)。

当 d=64, n=4096 时，nd² ≈ 16M 操作，而 n²d ≈ 1,073M 操作，线性注意力快 60 倍以上。

**代价：**
- 失去了非线性的指数 softmax，注意力分布更"软"，可能无法聚焦于尖锐的关键位置。  
- d×d 的上下文矩阵 C 成为了瓶颈，当 d 也很大时（如 d=1024），O(d²) 又会变大。  
- 但研究者们后续提出各种改进（如基于核函数的近似、混合注意力）来弥补这些不足。


## 5. 一路走来的思考

我们亲眼见证了解决 O(n²) 问题的两条路线：
- **结构稀疏化**（窗口、窗口+全局）：用人类先验强制丢弃远距离连接，简单有效，但需要精巧设计掩码模式。
- **代数线性化**（核函数分解）：用数学恒等式重组计算，从平方降到线性，几乎没有先验假设，但牺牲了注意力分布的锐度。

这两种思想至今仍在融合，例如在保留 softmax 的块稀疏注意力中采用代数优化，或在线性注意力中引入局部窗口以恢复锐度。你现在手中这些纯手工代码，就是这些现代长序列模型设计的基石。


## 6. 实例对比：四种注意力机制在真实句子上的表现

用一个具体的英文句子作为输入，在同一组随机 Embedding 和投影矩阵下，分别运行四种注意力机制，对比它们的注意力分布和稀疏性。

**输入句子**: `Lily is waling along the river bank and she saw a sign said "hello world" on it.`

我们关注三个查询词：**"Lily"**（句首）、**"saw"**（句中）、**"world"**（句尾），观察不同注意力机制下每个查询分别"注意"到了哪些键。

### 6.1 定义四种注意力函数
复用前文实现的四种注意力机制，封装为统一接口。

In [7]:
import math
import torch
import torch.nn.functional as F

def full_attention(Q, K, V):
    scale = math.sqrt(Q.shape[-1])
    attn = F.softmax((Q @ K.T) / scale, dim=-1)
    return attn, attn @ V

def window_attention(Q, K, V, w=3):
    n = Q.shape[0]
    scale = math.sqrt(Q.shape[-1])
    scores = (Q @ K.T) / scale
    mask = torch.ones(n, n)
    for i in range(n):
        for j in range(n):
            if abs(i - j) > w:
                mask[i, j] = 0
    attn = F.softmax(scores + (1 - mask) * -1e9, dim=-1)
    return attn, attn @ V

def hybrid_attention(Q, K, V, w=3, g=2):
    total = Q.shape[0]
    scale = math.sqrt(Q.shape[-1])
    scores = (Q @ K.T) / scale
    mask = torch.ones(total, total)
    for i in range(g, total):
        for j in range(g, total):
            if abs((i - g) - (j - g)) > w:
                mask[i, j] = 0
    attn = F.softmax(scores + (1 - mask) * -1e9, dim=-1)
    return attn, attn @ V

def linear_attention_fn(Q, K, V, eps=1e-6):
    phi_Q = F.elu(Q) + 1
    phi_K = F.elu(K) + 1
    C = phi_K.T @ V
    s = phi_K.T @ torch.ones(K.shape[0], 1)
    num = phi_Q @ C
    den = phi_Q @ s + eps
    out = num / den
    # 近似注意力权重（用于可视化，非严格 softmax）
    attn_approx = phi_Q @ phi_K.T
    attn_approx = attn_approx / attn_approx.sum(dim=-1, keepdim=True)
    return attn_approx, out


### 6.2 Tokenize 句子并生成 Embedding
将句子按空格分词，为每个词生成一个 64 维的随机向量（模拟词嵌入），再用随机投影矩阵计算 Q、K、V。

In [8]:
torch.manual_seed(42)
d_model, d_k = 64, 64
window_size, g = 3, 2

sentence = 'Lily is waling along the river bank and she saw a sign said "hello world" on it.'
token_strs = sentence.replace('"', ' ').split()
n = len(token_strs)
print('Tokens:', token_strs)
print('Total:', n)

X = torch.randn(n, d_model)
W_Q = torch.randn(d_model, d_k)
W_K = torch.randn(d_model, d_k)
W_V = torch.randn(d_model, d_k)

Q = F.normalize(X @ W_Q, dim=-1)
K = F.normalize(X @ W_K, dim=-1)
V = X @ W_V


Tokens: ['Lily', 'is', 'waling', 'along', 'the', 'river', 'bank', 'and', 'she', 'saw', 'a', 'sign', 'said', 'hello', 'world', 'on', 'it.']
Total: 17


### 6.3 运行四种注意力并对比注意力分布
分别运行全注意力、窗口注意力、混合注意力、线性注意力，打印每个查询的 Top-5 关注键及其权重。

In [9]:
# 全注意力
attn_full, out_full = full_attention(Q, K, V)

# 窗口注意力
attn_win, out_win = window_attention(Q, K, V, w=window_size)

# 混合注意力（拼接 2 个全局 token）
global_tokens = torch.randn(g, d_model, requires_grad=True)
X_hybrid = torch.cat([global_tokens, X], dim=0)
Q_h = F.normalize(X_hybrid @ W_Q, dim=-1)
K_h = F.normalize(X_hybrid @ W_K, dim=-1)
V_h = X_hybrid @ W_V
attn_hyb, out_hyb = hybrid_attention(Q_h, K_h, V_h, w=window_size, g=g)

# 线性注意力
attn_lin, out_lin = linear_attention_fn(Q, K, V)

hybrid_labels = ['[G0]', '[G1]'] + token_strs

query_map = {'Lily': 0, 'saw': 9, 'world': 14}
for name, attn, offset, labels in [
    ('Full Attention', attn_full, 0, token_strs),
    ('Window Attention', attn_win, 0, token_strs),
    ('Hybrid Attention', attn_hyb, g, hybrid_labels),
    ('Linear Attention', attn_lin, 0, token_strs),
]:
    print(f'\n── {name} ──')
    for qname, qi in query_map.items():
        qi_actual = qi + offset
        vals, idxs = attn[qi_actual].topk(5)
        pairs = [(labels[i.item()], round(v.item(), 3)) for i, v in zip(idxs, vals)]
        print(f'  Q["{qname}"] (pos {qi_actual}) top-5: {pairs}')



── Full Attention ──
  Q["Lily"] (pos 0) top-5: [('hello', 0.06), ('and', 0.06), ('it.', 0.06), ('a', 0.06), ('along', 0.06)]
  Q["saw"] (pos 9) top-5: [('it.', 0.061), ('waling', 0.06), ('saw', 0.059), ('along', 0.059), ('Lily', 0.059)]
  Q["world"] (pos 14) top-5: [('waling', 0.06), ('along', 0.06), ('said', 0.06), ('river', 0.06), ('saw', 0.06)]

── Window Attention ──
  Q["Lily"] (pos 0) top-5: [('along', 0.256), ('Lily', 0.252), ('is', 0.247), ('waling', 0.245), ('the', 0.0)]
  Q["saw"] (pos 9) top-5: [('saw', 0.145), ('sign', 0.143), ('she', 0.143), ('said', 0.143), ('bank', 0.142)]
  Q["world"] (pos 14) top-5: [('said', 0.17), ('on', 0.169), ('world', 0.166), ('it.', 0.165), ('hello', 0.165)]

── Hybrid Attention ──
  Q["Lily"] (pos 2) top-5: [('[G1]', 0.171), ('along', 0.17), ('Lily', 0.168), ('is', 0.164), ('[G0]', 0.164)]
  Q["saw"] (pos 11) top-5: [('[G1]', 0.114), ('saw', 0.112), ('sign', 0.111), ('she', 0.111), ('said', 0.111)]
  Q["world"] (pos 16) top-5: [('said', 0.127

### 6.4 稀疏性对比
计算每种注意力机制中非零权重（>1e-5）的比例，直观反映计算复杂度差异。

In [10]:
print('── Attention Sparsity (% non-zero weights) ──')
for name, attn, _, _ in [
    ('Full Attention', attn_full, 0, token_strs),
    ('Window Attention', attn_win, 0, token_strs),
    ('Hybrid Attention', attn_hyb, g, hybrid_labels),
    ('Linear Attention', attn_lin, 0, token_strs),
]:
    frac = (attn.detach().numpy() > 1e-5).mean()
    print(f'  {name:20s}  {frac:.1%}')

print()
print('分析:')
print('  - Full / Linear Attention: 100% 非零，每个查询注意所有键')
print('  - Window Attention: ~37% 非零，只注意窗口内邻居')
print('  - Hybrid Attention: ~50% 非零，比纯窗口多了全局 token 列')


── Attention Sparsity (% non-zero weights) ──
  Full Attention        100.0%
  Window Attention      37.0%
  Hybrid Attention      49.6%
  Linear Attention      100.0%

分析:
  - Full / Linear Attention: 100% 非零，每个查询注意所有键
  - Window Attention: ~37% 非零，只注意窗口内邻居
  - Hybrid Attention: ~50% 非零，比纯窗口多了全局 token 列


### 6.5 注意力热力图可视化

下图为四种注意力机制的注意力权重矩阵热力图，行 = 查询，列 = 键，颜色越深表示权重越大。

![Attention Comparison](attention_comparison.png)

**观察要点：**
- **Full Attention**：整个矩阵颜色均匀，每个查询均匀注意所有 17 个键，无结构偏好。
- **Window Attention**：对角线附近出现深色带状区域（窗口内），远处为白色（被 mask 屏蔽），稀疏性明显。
- **Hybrid Attention**：前两列（全局 token）始终有色，局部区域仍呈带状，窗口外局部位置被屏蔽。
- **Linear Attention**：矩阵整体均匀但数值偏低，因为 ELU+1 映射使注意力分布更"软"，缺乏锐利峰值。